In [34]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [35]:
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

In [36]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F

label_map = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}

#Load DeBERTa
deberta_tokenizer = AutoTokenizer.from_pretrained('microsoft/deberta-v3-small')
deberta_model = AutoModelForSequenceClassification.from_pretrained('microsoft/deberta-v3-small', num_labels=5)

#Load RoBERTa
roberta_tokenizer = AutoTokenizer.from_pretrained('roberta-base')
roberta_model = AutoModelForSequenceClassification.from_pretrained('roberta-base', num_labels=5)

deberta_model.eval()
roberta_model.eval()

print("Setup completed")

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.weight       

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Setup completed


In [37]:
#Q1 Answer A, 0.2122
row = train.iloc[25]
text = str(row['prompt'])

inputs_deberta = deberta_tokenizer(text, return_tensors='pt', truncation=True, max_length=128)
with torch.no_grad():
    outputs_deberta = deberta_model(**inputs_deberta)

probs_deberta = F.softmax(outputs_deberta.logits, dim=-1)[0]
best_idx = probs_deberta.argmax().item()
best_label = label_map[best_idx]
best_prob = round(probs_deberta[best_idx].item(), 4)

print(f"{best_label}, {best_prob}")

A, 0.2122


In [38]:
#Q2 Answer A, 0.2214
inputs_roberta = roberta_tokenizer(text, return_tensors='pt', truncation=True, max_length=128)
with torch.no_grad():
    outputs_roberta = roberta_model(**inputs_roberta)

probs_roberta = F.softmax(outputs_roberta.logits, dim=-1)[0]
avg_probs = (probs_deberta + probs_roberta) / 2

best_idx_avg = avg_probs.argmax().item()
print(f"{label_map[best_idx_avg]}, {round(avg_probs[best_idx_avg].item(), 4)}")

A, 0.2214


In [39]:
#Q3 Answer A, 0.2178
weighted_probs = (0.7 * probs_deberta) + (0.3 * probs_roberta)
best_idx_w = weighted_probs.argmax().item()
print(f"{label_map[best_idx_w]}, {round(weighted_probs[best_idx_w].item(), 4)}")

A, 0.2178


In [40]:
#Q4 Answer Top 3: A C D
all_ranked_idx = weighted_probs.argsort(descending=True)
all_ranked_labels = [label_map[i.item()] for i in all_ranked_idx]
print("All 5 ranked:", ' '.join(all_ranked_labels))
print("Top 3:", ' '.join(all_ranked_labels[:3]))

All 5 ranked: A C D E B
Top 3: A C D


In [41]:
#Q5 Answer Total prediction rows: 500
test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

predictions = []
for i in range(len(test)):
    row = test.iloc[i]
    text = str(row['prompt'])
    
    inputs_d = deberta_tokenizer(text, return_tensors='pt', truncation=True, max_length=128)
    with torch.no_grad():
        probs_d = F.softmax(deberta_model(**inputs_d).logits, dim=-1)[0]
    
    inputs_r = roberta_tokenizer(text, return_tensors='pt', truncation=True, max_length=128)
    with torch.no_grad():
        probs_r = F.softmax(roberta_model(**inputs_r).logits, dim=-1)[0]
        
    weighted = (0.7 * probs_d) + (0.3 * probs_r)
    top3 = ' '.join([label_map[i.item()] for i in weighted.argsort(descending=True)[:3]])
    predictions.append({'id': row['id'], 'prediction': top3})

submission = pd.DataFrame(predictions)
submission.to_csv('submission.csv', index=False)

print(f"Total prediction rows: {len(submission)}")

Total prediction rows: 500


In [42]:
#Q6 Answer Different Top-1 predictions after TTA: 24
different_count = 0
for i in range(50):
    row = test.iloc[i]
    text_original = str(row['prompt'])
    text_augmented = "Answer the following multiple-choice question carefully: " + text_original

    inputs_orig = deberta_tokenizer(text_original, return_tensors='pt', truncation=True, max_length=128)
    with torch.no_grad():
        probs_orig = F.softmax(deberta_model(**inputs_orig).logits, dim=-1)[0]

    inputs_aug = deberta_tokenizer(text_augmented, return_tensors='pt', truncation=True, max_length=128)
    with torch.no_grad():
        probs_aug = F.softmax(deberta_model(**inputs_aug).logits, dim=-1)[0]
        
    avg = (probs_orig + probs_aug) / 2

    top1_original = probs_orig.argmax().item()
    top1_tta = avg.argmax().item()

    if top1_original != top1_tta:
        different_count += 1

print(f"Different Top-1 predictions after TTA: {different_count}")

Different Top-1 predictions after TTA: 24


In [43]:
different_top1 = 0      # Q7 Answer 31 
positive_gain = 0        # Q8 Answer 7
different_top3 = 0       # Q9 Answer 65

for i in range(100):
    row = test.iloc[i]
    text = str(row['prompt'])

    inputs_d = deberta_tokenizer(text, return_tensors='pt', truncation=True, max_length=128)
    with torch.no_grad():
        probs_d = F.softmax(deberta_model(**inputs_d).logits, dim=-1)[0]

    inputs_r = roberta_tokenizer(text, return_tensors='pt', truncation=True, max_length=128)
    with torch.no_grad():
        probs_r = F.softmax(roberta_model(**inputs_r).logits, dim=-1)[0]

    weighted = (0.7 * probs_d) + (0.3 * probs_r)

    # Q7
    if probs_d.argmax().item() != weighted.argmax().item():
        different_top1 += 1

    # Q8
    if weighted.max().item() - probs_d.max().item() > 0:
        positive_gain += 1

    # Q9
    top3_deberta = [label_map[i.item()] for i in probs_d.argsort(descending=True)[:3]]
    top3_ensemble = [label_map[i.item()] for i in weighted.argsort(descending=True)[:3]]
    if top3_deberta != top3_ensemble:
        different_top3 += 1

print(f"Q7 - Different Top-1: {different_top1}")
print(f"Q8 - Positive confidence gain: {positive_gain}")
print(f"Q9 - Different Top-3: {different_top3}")

Q7 - Different Top-1: 31
Q8 - Positive confidence gain: 7
Q9 - Different Top-3: 65


In [44]:
#Q10 Answer MAP@3: 0.3833
label_map_reverse = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
map3_scores = []

for i in range(100):
    row = train.iloc[i]
    text = str(row['prompt'])
    correct = row['answer']

    inputs_d = deberta_tokenizer(text, return_tensors='pt', truncation=True, max_length=128)
    with torch.no_grad():
        probs_d = F.softmax(deberta_model(**inputs_d).logits, dim=-1)[0]

    inputs_r = roberta_tokenizer(text, return_tensors='pt', truncation=True, max_length=128)
    with torch.no_grad():
        probs_r = F.softmax(roberta_model(**inputs_r).logits, dim=-1)[0]

    weighted = (0.7 * probs_d) + (0.3 * probs_r)
    top3 = [label_map[i.item()] for i in weighted.argsort(descending=True)[:3]]

    score = next((1.0/(k+1) for k, l in enumerate(top3) if l == correct), 0.0)
    map3_scores.append(score)

print(f"Q10 - MAP@3: {round(np.mean(map3_scores), 4)}")

Q10 - MAP@3: 0.3833
